# 🚀 Distributed Matrix Multiplication — GPU Experiments
Run CUDA naïve, CUDA tiled, and OpenMP GPU kernels on Colab's free T4 GPU.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
!nvidia-smi
!nvcc --version

## 1. Core Utilities

In [ ]:
%%writefile matrix.h
#ifndef MATRIX_H
#define MATRIX_H
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <string.h>
#define MATRIX_SEED 42
#define VERIFY_TOLERANCE 1e-3f
#ifdef __cplusplus
extern "C" {
#endif
float* matrix_alloc(int rows, int cols);
void matrix_free(float* M);
void matrix_init_deterministic(float* M, int rows, int cols, int seed_offset);
void matrix_zero(float* M, int rows, int cols);
int matrix_verify(const float* expected, const float* actual, int rows, int cols);
void matrix_print(const float* M, int rows, int cols, const char* name);
#ifdef __cplusplus
}
#endif
#endif

In [ ]:
%%writefile matrix.c
#include "matrix.h"
float* matrix_alloc(int rows, int cols) {
    if (rows <= 0 || cols <= 0) return NULL;
    return (float*)calloc((size_t)rows * cols, sizeof(float));
}
void matrix_free(float* M) { if (M) free(M); }
void matrix_init_deterministic(float* M, int rows, int cols, int seed_offset) {
    srand(MATRIX_SEED + seed_offset);
    for (int i = 0; i < rows * cols; i++) M[i] = (float)rand() / (float)RAND_MAX * 10.0f;
}
void matrix_zero(float* M, int rows, int cols) { memset(M, 0, (size_t)rows * cols * sizeof(float)); }
int matrix_verify(const float* expected, const float* actual, int rows, int cols) {
    for (int i = 0; i < rows * cols; i++) {
        if (fabsf(expected[i] - actual[i]) > VERIFY_TOLERANCE) {
            fprintf(stderr, "FAIL at %d: expected=%.6f actual=%.6f\n", i, expected[i], actual[i]);
            return 0;
        }
    }
    return 1;
}
void matrix_print(const float* M, int rows, int cols, const char* name) {
    printf("\n=== %s (%dx%d) ===\n", name, rows, cols);
    for (int i = 0; i < rows; i++) {
        for (int j = 0; j < cols; j++) printf("%8.3f", M[i*cols+j]);
        printf("\n");
    }
}

## 2. CPU Sequential Baseline

In [ ]:
%%writefile cpu_sequential.cu
#include <stdio.h>
#include <time.h>
#include "matrix.h"

double get_time_ms() { struct timespec ts; clock_gettime(CLOCK_MONOTONIC, &ts); return ts.tv_sec*1000.0 + ts.tv_nsec/1e6; }

void cpu_matmul(const float* A, const float* B, float* C, int N) {
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++) sum += A[i*N+k] * B[k*N+j];
            C[i*N+j] = sum;
        }
}

int main(int argc, char** argv) {
    int N = argc > 1 ? atoi(argv[1]) : 512;
    float *A = matrix_alloc(N,N), *B = matrix_alloc(N,N), *C = matrix_alloc(N,N);
    matrix_init_deterministic(A, N, N, 0);
    matrix_init_deterministic(B, N, N, 1);
    double t0 = get_time_ms();
    cpu_matmul(A, B, C, N);
    double t1 = get_time_ms();
    double gflops = 2.0*N*N*N / ((t1-t0)/1000.0) / 1e9;
    printf("CPU,%d,%.3f,%.4f,PASS\n", N, t1-t0, gflops);
    matrix_free(A); matrix_free(B); matrix_free(C);
    return 0;
}

In [ ]:
!nvcc -o cpu_seq cpu_sequential.cu matrix.c -lm
!echo '--- 256 ---' && ./cpu_seq 256
!echo '--- 512 ---' && ./cpu_seq 512
!echo '--- 1024 ---' && ./cpu_seq 1024

## 3. CUDA Naïve Kernel

In [ ]:
%%writefile cuda_naive.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include "matrix.h"

#define BLOCK 16
#define CUDA_CHECK(call) { cudaError_t e=(call); if(e!=cudaSuccess){fprintf(stderr,"CUDA err %s:%d %s\n",__FILE__,__LINE__,cudaGetErrorString(e));exit(1);} }

__global__ void naive_kernel(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++) sum += A[row*N+k] * B[k*N+col];
        C[row*N+col] = sum;
    }
}

void cpu_ref(const float* A, const float* B, float* C, int N) {
    for(int i=0;i<N;i++) for(int j=0;j<N;j++) { float s=0; for(int k=0;k<N;k++) s+=A[i*N+k]*B[k*N+j]; C[i*N+j]=s; }
}

int main(int argc, char** argv) {
    int N = argc > 1 ? atoi(argv[1]) : 512;
    size_t bytes = N*N*sizeof(float);
    float *hA=matrix_alloc(N,N), *hB=matrix_alloc(N,N), *hC=matrix_alloc(N,N), *hRef=matrix_alloc(N,N);
    matrix_init_deterministic(hA,N,N,0); matrix_init_deterministic(hB,N,N,1);
    float *dA,*dB,*dC;
    CUDA_CHECK(cudaMalloc(&dA,bytes)); CUDA_CHECK(cudaMalloc(&dB,bytes)); CUDA_CHECK(cudaMalloc(&dC,bytes));
    CUDA_CHECK(cudaMemcpy(dA,hA,bytes,cudaMemcpyHostToDevice)); CUDA_CHECK(cudaMemcpy(dB,hB,bytes,cudaMemcpyHostToDevice));
    dim3 block(BLOCK,BLOCK), grid((N+BLOCK-1)/BLOCK,(N+BLOCK-1)/BLOCK);
    cudaEvent_t s,e; cudaEventCreate(&s); cudaEventCreate(&e);
    cudaEventRecord(s); naive_kernel<<<grid,block>>>(dA,dB,dC,N); cudaEventRecord(e); cudaDeviceSynchronize();
    float ms; cudaEventElapsedTime(&ms,s,e);
    CUDA_CHECK(cudaMemcpy(hC,dC,bytes,cudaMemcpyDeviceToHost));
    cpu_ref(hA,hB,hRef,N);
    int ok = matrix_verify(hRef,hC,N,N);
    double gf = 2.0*N*N*N/((double)ms/1000.0)/1e9;
    printf("CUDA_Naive,%d,%.3f,%.4f,%s\n", N, ms, gf, ok?"PASS":"FAIL");
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    matrix_free(hA); matrix_free(hB); matrix_free(hC); matrix_free(hRef);
}

In [ ]:
!nvcc -o cuda_naive cuda_naive.cu matrix.c -lm
!echo '--- 256 ---' && ./cuda_naive 256
!echo '--- 512 ---' && ./cuda_naive 512
!echo '--- 1024 ---' && ./cuda_naive 1024
!echo '--- 2048 ---' && ./cuda_naive 2048

## 4. CUDA Tiled Kernel (Shared Memory)

In [ ]:
%%writefile cuda_tiled.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include "matrix.h"

#define TILE 16
#define CUDA_CHECK(call) { cudaError_t e=(call); if(e!=cudaSuccess){fprintf(stderr,"CUDA err %s:%d %s\n",__FILE__,__LINE__,cudaGetErrorString(e));exit(1);} }

__global__ void tiled_kernel(const float* A, const float* B, float* C, int N) {
    __shared__ float tA[TILE][TILE], tB[TILE][TILE];
    int row = blockIdx.y*TILE + threadIdx.y, col = blockIdx.x*TILE + threadIdx.x;
    float sum = 0.0f;
    for (int t = 0; t < (N+TILE-1)/TILE; t++) {
        int ac = t*TILE+threadIdx.x, br = t*TILE+threadIdx.y;
        tA[threadIdx.y][threadIdx.x] = (row<N && ac<N) ? A[row*N+ac] : 0.0f;
        tB[threadIdx.y][threadIdx.x] = (br<N && col<N) ? B[br*N+col] : 0.0f;
        __syncthreads();
        for (int k=0; k<TILE; k++) sum += tA[threadIdx.y][k] * tB[k][threadIdx.x];
        __syncthreads();
    }
    if (row<N && col<N) C[row*N+col] = sum;
}

void cpu_ref(const float* A, const float* B, float* C, int N) {
    for(int i=0;i<N;i++) for(int j=0;j<N;j++) { float s=0; for(int k=0;k<N;k++) s+=A[i*N+k]*B[k*N+j]; C[i*N+j]=s; }
}

int main(int argc, char** argv) {
    int N = argc > 1 ? atoi(argv[1]) : 512;
    size_t bytes = N*N*sizeof(float);
    float *hA=matrix_alloc(N,N), *hB=matrix_alloc(N,N), *hC=matrix_alloc(N,N), *hRef=matrix_alloc(N,N);
    matrix_init_deterministic(hA,N,N,0); matrix_init_deterministic(hB,N,N,1);
    float *dA,*dB,*dC;
    CUDA_CHECK(cudaMalloc(&dA,bytes)); CUDA_CHECK(cudaMalloc(&dB,bytes)); CUDA_CHECK(cudaMalloc(&dC,bytes));
    CUDA_CHECK(cudaMemcpy(dA,hA,bytes,cudaMemcpyHostToDevice)); CUDA_CHECK(cudaMemcpy(dB,hB,bytes,cudaMemcpyHostToDevice));
    dim3 block(TILE,TILE), grid((N+TILE-1)/TILE,(N+TILE-1)/TILE);
    // Print device info
    cudaDeviceProp prop; cudaGetDeviceProperties(&prop,0);
    printf("Device: %s | SharedMem/Block: %zu bytes | Tile: %dx%d\n", prop.name, prop.sharedMemPerBlock, TILE, TILE);
    cudaEvent_t s,e; cudaEventCreate(&s); cudaEventCreate(&e);
    cudaEventRecord(s); tiled_kernel<<<grid,block>>>(dA,dB,dC,N); cudaEventRecord(e); cudaDeviceSynchronize();
    float ms; cudaEventElapsedTime(&ms,s,e);
    CUDA_CHECK(cudaMemcpy(hC,dC,bytes,cudaMemcpyDeviceToHost));
    cpu_ref(hA,hB,hRef,N);
    int ok = matrix_verify(hRef,hC,N,N);
    double gf = 2.0*N*N*N/((double)ms/1000.0)/1e9;
    printf("CUDA_Tiled,%d,%.3f,%.4f,%s\n", N, ms, gf, ok?"PASS":"FAIL");
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    matrix_free(hA); matrix_free(hB); matrix_free(hC); matrix_free(hRef);
}

In [ ]:
!nvcc -o cuda_tiled cuda_tiled.cu matrix.c -lm
!echo '--- 256 ---' && ./cuda_tiled 256
!echo '--- 512 ---' && ./cuda_tiled 512
!echo '--- 1024 ---' && ./cuda_tiled 1024
!echo '--- 2048 ---' && ./cuda_tiled 2048

## 5. Edge Case Testing

In [ ]:
!echo '=== Edge Cases (non-multiple of tile) ==='
!./cuda_naive 1 && ./cuda_tiled 1
!./cuda_naive 15 && ./cuda_tiled 15
!./cuda_naive 17 && ./cuda_tiled 17
!./cuda_naive 33 && ./cuda_tiled 33
!./cuda_naive 255 && ./cuda_tiled 255
!./cuda_naive 257 && ./cuda_tiled 257

## 6. Benchmark & Charts

In [ ]:
import subprocess, csv, io
import matplotlib.pyplot as plt
import numpy as np

sizes = [256, 512, 1024, 2048]
results = {'CPU':[], 'CUDA_Naive':[], 'CUDA_Tiled':[]}

for N in sizes:
    for prog, key in [('./cpu_seq','CPU'), ('./cuda_naive','CUDA_Naive'), ('./cuda_tiled','CUDA_Tiled')]:
        out = subprocess.run([prog, str(N)], capture_output=True, text=True).stdout.strip().split('\n')[-1]
        parts = out.split(',')
        results[key].append({'size':N, 'time':float(parts[2]), 'gflops':float(parts[3]), 'ok':parts[4]})
        print(out)

print('\nAll benchmarks complete!')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#1a1a2e')
colors = {'CPU':'#6366f1', 'CUDA_Naive':'#ef4444', 'CUDA_Tiled':'#f97316'}

# Chart 1: Execution Time
ax = axes[0]; ax.set_facecolor('#16213e')
x = np.arange(len(sizes)); w = 0.25
for i,(k,v) in enumerate(results.items()):
    ax.bar(x+i*w, [r['time'] for r in v], w, label=k, color=colors[k])
ax.set_yscale('log'); ax.set_xticks(x+w); ax.set_xticklabels(sizes)
ax.set_xlabel('Matrix Size',color='white'); ax.set_ylabel('Time (ms)',color='white')
ax.set_title('Execution Time',color='white'); ax.legend(); ax.tick_params(colors='white')

# Chart 2: Speedup
ax = axes[1]; ax.set_facecolor('#16213e')
for k in ['CUDA_Naive','CUDA_Tiled']:
    speedups = [results['CPU'][i]['time']/results[k][i]['time'] for i in range(len(sizes))]
    ax.plot(sizes, speedups, 'o-', label=k, color=colors[k], linewidth=2)
ax.axhline(1, color='white', ls='--', alpha=0.3)
ax.set_xlabel('Matrix Size',color='white'); ax.set_ylabel('Speedup (x)',color='white')
ax.set_title('Speedup vs CPU',color='white'); ax.legend(); ax.tick_params(colors='white')

# Chart 3: GFLOPS
ax = axes[2]; ax.set_facecolor('#16213e')
for i,(k,v) in enumerate(results.items()):
    ax.bar(x+i*w, [r['gflops'] for r in v], w, label=k, color=colors[k])
ax.set_xticks(x+w); ax.set_xticklabels(sizes)
ax.set_xlabel('Matrix Size',color='white'); ax.set_ylabel('GFLOPS',color='white')
ax.set_title('Throughput (GFLOPS)',color='white'); ax.legend(); ax.tick_params(colors='white')

plt.tight_layout(); plt.savefig('gpu_benchmark_charts.png', dpi=150, facecolor='#1a1a2e')
plt.show(); print('Charts saved to gpu_benchmark_charts.png')

## 7. Export Results

In [ ]:
# Save CSV
with open('gpu_timings.csv','w') as f:
    f.write('mode,matrix_size,exec_time_ms,gflops,verified\n')
    for k,v in results.items():
        for r in v: f.write(f"{k},{r['size']},{r['time']},{r['gflops']},{r['ok']}\n")
print('Saved gpu_timings.csv')

# Download
from google.colab import files
files.download('gpu_timings.csv')
files.download('gpu_benchmark_charts.png')